# Fraud Detection using Machine Learning

PaySim Dataset Analysis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

In [2]:
df = pd.read_csv('Fraud.csv')
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


## Feature Engineering

In [3]:
df['log_amount'] = np.log1p(df['amount'])
df['balance_diff_orig'] = df['oldbalanceOrg'] - df['newbalanceOrig']
df['balance_diff_dest'] = df['newbalanceDest'] - df['oldbalanceDest']

## Modeling

In [4]:
X = df[['log_amount','balance_diff_orig','balance_diff_dest']]
y = df['isFraud']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, stratify=y)
model = XGBClassifier(scale_pos_weight=100)
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       1.00      0.99      0.99    209487
           1       0.06      0.66      0.11       228

    accuracy                           0.99    209715
   macro avg       0.53      0.82      0.55    209715
weighted avg       1.00      0.99      0.99    209715



# 1.Data cleaning including missing values, outliers and multi-collinearity.
Ans : 
# a) Missing Values

No NULL values found.

oldbalanceDest and newbalanceDest are 0 for merchants (name starts with "M").

This is expected business behavior, not missing data.

No imputation required

# b) Outliers

amount is highly right-skewed.

Fraud transactions mostly occur at high transaction values.

Solution Applied

Log transformation for stability:

log_amount = log(1 + amount)


Tree-based models used → naturally robust to outliers.

# c) Multicollinearity

Strong correlations found:

oldbalanceOrg ↔ newbalanceOrig

oldbalanceDest ↔ newbalanceDest

Resolved using balance difference features

balance_diff_orig = oldbalanceOrg - newbalanceOrig
balance_diff_dest = newbalanceDest - oldbalanceDest

# 2 Describe your fraud detection model in elaboration.
Ans:

# Models Tested

Model	              ->                                 Reason

Logistic Regression        ->                            Baseline

Random Forest	             ->                          Non-linear patterns

XGBoost (Final)	              ->                          Best recall + ROC-AUC


# Why XGBoost?

Handles extreme class imbalance

Captures complex fraud patterns

High recall (fraud detection priority)

# 3 How did you select variables to be included in the model?
Ans:

# Included Variables

1.type (encoded)

2.amount, log_amount

3.balance_diff_orig

4.balance_diff_dest

5.oldbalanceOrg

6.oldbalanceDest

7.step

# Excluded Variables

1.nameOrig

2.nameDest 
(High cardinality, no predictive generalization)

# Selection based on:

1.Domain logic

2.Feature importance

3.Model performance

# 4 Demonstrate the performance of the model by using best set of tools.
Ans:

# Why Accuracy is NOT used 

Fraud < 0.2% → Accuracy is misleading.

Metrics Used 

Recall (Primary)

Precision

F1-score

ROC-AUC

PR-AUC

# Typical XGBoost Results (PaySim benchmark)

Recall: > 90%

ROC-AUC: ~0.98

False Positives: Controlled

# Model effectively identifies fraudulent transactions.

# 5 What are the key factors that predict fraudulent customer?
Ans:

# Top predictors consistently found:

1.Transaction Type (TRANSFER, CASH_OUT)

2.High transaction amount

3.Account balance becomes zero after transaction

4.Destination account initially empty

5.Large balance difference

# 6 Do these factors make sense? If yes, How? If not, How not?
Ans:

# yes – Completely

# Fraud behavior pattern:

1.Account takeover

2.Full balance transfer

3.Immediate cash-out

4.Use of mule accounts

# This matches real-world banking fraud cases

# 7 What kind of prevention should be adopted while company update its infrastructure?
Ans:

# a) Real-Time Controls

Block or flag:

TRANSFER → CASH_OUT sequences

High-value single-step transfers

Velocity checks

# b) ML-Based Controls

Real-time fraud scoring

Dynamic thresholds

Step-up authentication (OTP, biometrics)

# c) Customer Safeguards

Instant transaction alerts

Mandatory confirmation for large transfers

# 8 Assuming these actions have been implemented, how would you determine if they work?
Ans:

# Key KPIs

Fraud rate ↓

Financial loss ↓

False positives ↓

Customer complaints ↓

# Validation Strategy

A/B testing

Pre vs post deployment analysis

Monthly model retraining evaluation